The StudentLife RDS files are available locally on my computer, but they are ignored from Git because they are too large.

Please update studentlife_bridge_model_analysis.ipynb so it can run locally using data/studentlife_rds/ or data/studentlife_rds/dataset_rds/.

Do not try to download StudentLife from Zenodo.

Search recursively for:
- phonelock.Rds
- Sleep.Rds
- Mood.Rds
- PerceivedStressScale.Rds
- Stress.Rds

Test:

A. screen time / phone use → sleep quality
Use phonelock.Rds + Sleep.Rds.

B. sleep quality → mood
Use Sleep.Rds + Mood.Rds.

C. sleep quality → stress
Use Sleep.Rds + PerceivedStressScale.Rds or Stress.Rds.

For each:
- rows used
- target variable
- predictors
- baseline mean model
- linear regression
- one stronger comparison model if enough rows
- train R²
- test R²
- test MAE
- test RMSE
- usable yes/no

Do not use sleep_score as evidence.
Do not invent connections.
Only mark a path as supported if the tested model beats baseline.

Update model_results_summary.md with:
StudentLife Bridge Tests — Actual Local Results

# StudentLife Bridge Model Analysis

This notebook tests whether StudentLife can support the sleep-quality bridge in the installation.

The fixed installation inputs are:

coffee
exercise
screen time

The fixed installation outputs are:

memory
reaction time / processing speed
mood
stress

This notebook focuses on the StudentLife paths:

screen time / phone use → sleep quality
sleep quality → mood
sleep quality → stress

Important rule:

The notebook does not use sleep_score as evidence.

A path is only marked usable if the tested model beats the baseline mean model on held-out test data.

In [1]:
# ------------------------------------------------------------
# StudentLife bridge analysis setup
# ------------------------------------------------------------

from pathlib import Path
import subprocess
import sys
import zipfile
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Install needed packages if missing
# ------------------------------------------------------------

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name

    try:
        __import__(import_name)
    except ImportError:
        print(f"{package_name} is missing. Installing it now...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])


install_if_missing("pyreadr")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("matplotlib")
install_if_missing("pandas")
install_if_missing("numpy")

import numpy as np
import pandas as pd
import pyreadr
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# Re-create paths
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
STUDENTLIFE_DIR = DATA_DIR / "studentlife_rds"
GRAPH_DIR = PROJECT_DIR / "Graphs"

STUDENTLIFE_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:")
print(PROJECT_DIR)

print("\nStudentLife data folder:")
print(STUDENTLIFE_DIR)

print("\nGraphs folder:")
print(GRAPH_DIR)

Project folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis

StudentLife data folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\studentlife_rds

Graphs folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\Graphs


## 1. Find and Load StudentLife RDS Files

This step searches locally for the StudentLife files.

The important files are:

phonelock.Rds
Sleep.Rds
Mood.Rds
PerceivedStressScale.Rds
Stress.Rds

If the files are inside dataset_rds.zip, the notebook will extract them first.

In [2]:
# ------------------------------------------------------------
# Find and load StudentLife RDS files
# ------------------------------------------------------------

# Extract local StudentLife ZIP if present
zip_candidates = list(STUDENTLIFE_DIR.rglob("dataset_rds.zip")) + list(DATA_DIR.rglob("dataset_rds.zip"))

if len(zip_candidates) > 0:
    zip_path = zip_candidates[0]
    print("Found StudentLife ZIP:")
    print(zip_path)

    extract_folder = STUDENTLIFE_DIR / "dataset_rds"
    extract_folder.mkdir(parents=True, exist_ok=True)

    # Extract only if folder looks empty
    existing_rds = list(extract_folder.rglob("*.Rds")) + list(extract_folder.rglob("*.rds"))

    if len(existing_rds) == 0:
        print("\nExtracting StudentLife ZIP...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_folder)
        print("Extraction finished.")
    else:
        print("\nRDS files already extracted. Skipping extraction.")
else:
    print("No dataset_rds.zip found. Searching for already extracted RDS files.")


# Search recursively for RDS files
rds_files = list(DATA_DIR.rglob("*.Rds")) + list(DATA_DIR.rglob("*.rds"))

print(f"\nTotal RDS files found: {len(rds_files)}")

# Required files
wanted_names = [
    "phonelock.Rds",
    "Sleep.Rds",
    "Mood.Rds",
    "PerceivedStressScale.Rds",
    "Stress.Rds",
    "PAM.Rds"
]

found_files = {}

for wanted_name in wanted_names:
    matches = [
        file_path for file_path in rds_files
        if file_path.name.lower() == wanted_name.lower()
    ]

    if len(matches) > 0:
        found_files[wanted_name] = matches[0]
    else:
        found_files[wanted_name] = None

found_table = pd.DataFrame({
    "file": list(found_files.keys()),
    "found_path": [
        str(path) if path is not None else "NOT FOUND"
        for path in found_files.values()
    ]
})

display(found_table)


# ------------------------------------------------------------
# Helper: load RDS file
# ------------------------------------------------------------

def load_rds(path):
    """
    Load an RDS file using pyreadr.
    Returns the first dataframe found inside the RDS object.
    """

    result = pyreadr.read_r(str(path))

    if len(result.keys()) == 0:
        raise ValueError(f"No object found inside {path}")

    first_key = list(result.keys())[0]
    df = result[first_key]

    return df.copy()


loaded = {}

for file_name, file_path in found_files.items():
    if file_path is not None:
        try:
            loaded[file_name] = load_rds(file_path)
            print(f"Loaded {file_name}: {loaded[file_name].shape[0]} rows × {loaded[file_name].shape[1]} columns")
        except Exception as error:
            print(f"Could not load {file_name}: {error}")

Found StudentLife ZIP:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\dataset_rds.zip

RDS files already extracted. Skipping extraction.

Total RDS files found: 100


,file,found_path
0,phonelock.Rds,D:\Documentos\Sleep datasets\sleep_installatio...
1,Sleep.Rds,D:\Documentos\Sleep datasets\sleep_installatio...
2,Mood.Rds,D:\Documentos\Sleep datasets\sleep_installatio...
3,PerceivedStressScale.Rds,D:\Documentos\Sleep datasets\sleep_installatio...
4,Stress.Rds,D:\Documentos\Sleep datasets\sleep_installatio...
5,PAM.Rds,D:\Documentos\Sleep datasets\sleep_installatio...


Loaded phonelock.Rds: 9275 rows × 3 columns
Loaded Sleep.Rds: 1644 rows × 6 columns
Loaded Mood.Rds: 277 rows × 7 columns
Loaded PerceivedStressScale.Rds: 85 rows × 12 columns
Loaded Stress.Rds: 2017 rows × 3 columns
Loaded PAM.Rds: 9040 rows × 3 columns


## 2. Inspect StudentLife Files

This step prints the columns and first rows of each loaded file.

This is important because we only use variables that actually exist in the files.

In [3]:
# ------------------------------------------------------------
# Inspect loaded StudentLife files
# ------------------------------------------------------------

for file_name, df in loaded.items():
    print("\n" + "=" * 80)
    print(file_name)
    print("=" * 80)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print("Columns:")
    print(list(df.columns))

    display(df.head())


phonelock.Rds
Shape: 9275 rows × 3 columns
Columns:
['start_timestamp', 'end_timestamp', 'uid']


,start_timestamp,end_timestamp,uid
0,1.364359e+09,1.364365e+09,0
1,1.364366e+09,1.364382e+09,0
2,1.364386e+09,1.364391e+09,0
3,1.364406e+09,1.364411e+09,0
4,1.364426e+09,1.364429e+09,0



Sleep.Rds
Shape: 1644 rows × 6 columns
Columns:
['hour', 'location', 'rate', 'timestamp', 'social', 'uid']


,hour,location,rate,timestamp,social,uid
0,NaN,NaN,NaN,1.364115e+09,NaN,0
1,NaN,NaN,NaN,1.364115e+09,NaN,0
2,NaN,NaN,NaN,1.364114e+09,NaN,0
3,NaN,NaN,NaN,1.364115e+09,NaN,0
4,NaN,NaN,NaN,1.364114e+09,NaN,0



Mood.Rds
Shape: 277 rows × 7 columns
Columns:
['happy', 'happyornot', 'location', 'timestamp', 'sad', 'sadornot', 'uid']


...

,uid,type,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10
0,0,pre,Sometime,Sometime,Fairly often,Fairly often,Sometime,Very often,Sometime,Sometime,Fairly often,Fairly often
1,1,pre,Sometime,Sometime,Sometime,Sometime,Fairly often,Sometime,Fairly often,Fairly often,Almost never,Almost never
2,2,pre,Fairly often,Sometime,Sometime,Fairly often,Almost never,Sometime,Almost never,Sometime,Sometime,Almost never
3,3,pre,Sometime,Almost never,Sometime,Almost never,Sometime,Never,Almost never,Never,Never,Never
4,4,pre,Almost never,Almost never,Fairly often,Sometime,Sometime,Fairly often,Sometime,Almost never,Sometime,Sometime



Stress.Rds
Shape: 2017 rows × 3 columns
Columns:
['null', 'timestamp', 'uid']


,null,timestamp,uid
0,"43.75908069,-72.32885314",1.364114e+09,0
1,"43.75908069,-72.32885314",1.364114e+09,0
2,"43.70677151,-72.28746626",1.364178e+09,0
3,1,1.364170e+09,0
4,1,1.364178e+09,0



PAM.Rds
Shape: 9040 rows × 3 columns
Columns:
['picture_idx', 'timestamp', 'uid']


,picture_idx,timestamp,uid
0,7.0,1.364115e+09,0
1,11.0,1.364156e+09,0
2,7.0,1.364114e+09,0
3,8.0,1.364115e+09,0
4,7.0,1.364264e+09,0


## 3. Prepare Helper Functions

This step creates reusable functions for:

finding user IDs
finding timestamps
making daily or user-level summaries
running baseline and regression models
saving graphs

The model rule is strict:

A path is usable only if the best model has lower test RMSE than the baseline mean model.

In [4]:
# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def normalize_column_name(column):
    return str(column).strip().lower().replace(" ", "_").replace("-", "_")


def find_column(df, possible_names=None, contains_any=None):
    """
    Find one column by exact possible names or by partial text.
    """

    if possible_names is None:
        possible_names = []

    if contains_any is None:
        contains_any = []

    normalized_lookup = {
        normalize_column_name(column): column
        for column in df.columns
    }

    # Exact name matching first
    for name in possible_names:
        clean_name = normalize_column_name(name)
        if clean_name in normalized_lookup:
            return normalized_lookup[clean_name]

    # Partial matching second
    for column in df.columns:
        clean_column = normalize_column_name(column)

        for pattern in contains_any:
            clean_pattern = normalize_column_name(pattern)

            if clean_pattern in clean_column:
                return column

    return None


def smart_to_datetime(series):
    """
    Convert a column into datetime as safely as possible.
    Handles normal strings and Unix timestamps.
    """

    # First try normal parsing
    parsed = pd.to_datetime(series, errors="coerce")

    if parsed.notna().mean() > 0.5:
        return parsed

    # Try Unix seconds
    parsed_seconds = pd.to_datetime(series, errors="coerce", unit="s")

    if parsed_seconds.notna().mean() > 0.5:
        return parsed_seconds

    # Try Unix milliseconds
    parsed_milliseconds = pd.to_datetime(series, errors="coerce", unit="ms")

    if parsed_milliseconds.notna().mean() > 0.5:
        return parsed_milliseconds

    return parsed


def guess_uid_column(df):
    return find_column(
        df,
        possible_names=["uid", "user_id", "participant_id", "id"],
        contains_any=["uid", "user"]
    )


def guess_time_column(df):
    return find_column(
        df,
        possible_names=["timestamp", "time", "datetime", "date"],
        contains_any=["timestamp", "time", "date"]
    )


def add_date_column(df, time_col):
    output = df.copy()

    if time_col is None:
        output["date"] = np.nan
        return output

    output["_datetime"] = smart_to_datetime(output[time_col])
    output["date"] = output["_datetime"].dt.date

    return output


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def evaluate_regression_path(
    path_name,
    data,
    predictors,
    target,
    output_label,
    graph_prefix
):
    """
    Evaluate baseline, linear regression, and random forest.

    A path is marked usable if best test RMSE is lower than baseline test RMSE.
    """

    model_data = data[predictors + [target]].copy()

    for column in predictors + [target]:
        model_data[column] = safe_numeric(model_data[column])

    model_data = model_data.dropna().copy()

    if len(model_data) < 20:
        return {
            "path": path_name,
            "output": output_label,
            "target": target,
            "rows": len(model_data),
            "baseline_r2": np.nan,
            "baseline_mae": np.nan,
            "baseline_rmse": np.nan,
            "linear_train_r2": np.nan,
            "linear_test_r2": np.nan,
            "linear_mae": np.nan,
            "linear_rmse": np.nan,
            "comparison_model": "not tested",
            "comparison_train_r2": np.nan,
            "comparison_test_r2": np.nan,
            "comparison_mae": np.nan,
            "comparison_rmse": np.nan,
            "best_model": "not enough rows",
            "beats_baseline": "No",
            "decision": "do not use",
            "graph_path": ""
        }

    X = model_data[predictors]
    y = model_data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42
    )

    # Baseline
    baseline = DummyRegressor(strategy="mean")
    baseline.fit(X_train, y_train)

    baseline_train_predictions = baseline.predict(X_train)
    baseline_test_predictions = baseline.predict(X_test)

    baseline_r2 = r2_score(y_test, baseline_test_predictions)
    baseline_mae = mean_absolute_error(y_test, baseline_test_predictions)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_test_predictions))

    # Linear regression
    linear_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    linear_model.fit(X_train, y_train)

    linear_train_predictions = linear_model.predict(X_train)
    linear_test_predictions = linear_model.predict(X_test)

    linear_train_r2 = r2_score(y_train, linear_train_predictions)
    linear_test_r2 = r2_score(y_test, linear_test_predictions)
    linear_mae = mean_absolute_error(y_test, linear_test_predictions)
    linear_rmse = np.sqrt(mean_squared_error(y_test, linear_test_predictions))

    # Random forest comparison
    random_forest = RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=4,
        random_state=42
    )

    random_forest.fit(X_train, y_train)

    rf_train_predictions = random_forest.predict(X_train)
    rf_test_predictions = random_forest.predict(X_test)

    rf_train_r2 = r2_score(y_train, rf_train_predictions)
    rf_test_r2 = r2_score(y_test, rf_test_predictions)
    rf_mae = mean_absolute_error(y_test, rf_test_predictions)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_test_predictions))

    # Pick best model by RMSE
    model_scores = [
        ("baseline mean", baseline_rmse, baseline_test_predictions),
        ("linear regression", linear_rmse, linear_test_predictions),
        ("random forest", rf_rmse, rf_test_predictions)
    ]

    best_model_name, best_rmse, best_predictions = sorted(
        model_scores,
        key=lambda item: item[1]
    )[0]

    beats_baseline = best_rmse < baseline_rmse

    decision = "use with caution" if beats_baseline and best_model_name != "baseline mean" else "do not use"

    # Save graph
    graph_path = GRAPH_DIR / f"{graph_prefix}_actual_vs_predicted.png"

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, best_predictions, alpha=0.7)
    plt.xlabel(f"Actual {target}")
    plt.ylabel(f"Predicted {target}")
    plt.title(f"{path_name}: actual vs predicted")
    plt.grid(True, alpha=0.3)

    min_value = min(y_test.min(), np.min(best_predictions))
    max_value = max(y_test.max(), np.max(best_predictions))
    plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

    plt.tight_layout()
    plt.savefig(graph_path, dpi=200)
    plt.close()

    return {
        "path": path_name,
        "output": output_label,
        "target": target,
        "rows": len(model_data),
        "baseline_r2": round(baseline_r2, 4),
        "baseline_mae": round(baseline_mae, 4),
        "baseline_rmse": round(baseline_rmse, 4),
        "linear_train_r2": round(linear_train_r2, 4),
        "linear_test_r2": round(linear_test_r2, 4),
        "linear_mae": round(linear_mae, 4),
        "linear_rmse": round(linear_rmse, 4),
        "comparison_model": "random forest",
        "comparison_train_r2": round(rf_train_r2, 4),
        "comparison_test_r2": round(rf_test_r2, 4),
        "comparison_mae": round(rf_mae, 4),
        "comparison_rmse": round(rf_rmse, 4),
        "best_model": best_model_name,
        "beats_baseline": "Yes" if beats_baseline else "No",
        "decision": decision,
        "graph_path": str(graph_path)
    }

## 4. Prepare Sleep, Screen, Mood, and Stress Variables

This step converts the raw StudentLife files into analysis tables.

The intended bridges are:

screen use → sleep quality
sleep quality → mood
sleep quality → stress

The notebook uses daily-level joins when dates are available.

If daily joining is not possible, it falls back to user-level summaries.

In [5]:
# ------------------------------------------------------------
# Prepare StudentLife variables
# ------------------------------------------------------------

# ------------------------------------------------------------
# Sleep.Rds
# ------------------------------------------------------------

sleep_daily = None
sleep_user = None

if "Sleep.Rds" in loaded:
    sleep_raw = loaded["Sleep.Rds"].copy()

    uid_col = guess_uid_column(sleep_raw)
    time_col = guess_time_column(sleep_raw)

    rate_col = find_column(
        sleep_raw,
        possible_names=["rate"],
        contains_any=["rate", "quality"]
    )

    hour_col = find_column(
        sleep_raw,
        possible_names=["hour"],
        contains_any=["hour", "duration", "sleep"]
    )

    print("Sleep columns detected:")
    print("uid:", uid_col)
    print("time:", time_col)
    print("rate:", rate_col)
    print("hour:", hour_col)

    sleep_prepared = sleep_raw.copy()

    if uid_col is not None:
        sleep_prepared["uid_clean"] = sleep_prepared[uid_col].astype(str)

    if time_col is not None:
        sleep_prepared = add_date_column(sleep_prepared, time_col)

    if rate_col is not None:
        sleep_prepared["sleep_rate_raw"] = safe_numeric(sleep_prepared[rate_col])

        # Earlier project logic treated higher rate as more sleep problem.
        sleep_prepared["sleep_quality_problem"] = sleep_prepared["sleep_rate_raw"] - sleep_prepared["sleep_rate_raw"].min()

    if hour_col is not None:
        sleep_prepared["sleep_hours"] = safe_numeric(sleep_prepared[hour_col])

    sleep_value_columns = [
        column for column in ["sleep_rate_raw", "sleep_quality_problem", "sleep_hours"]
        if column in sleep_prepared.columns
    ]

    if uid_col is not None and "date" in sleep_prepared.columns:
        sleep_daily = (
            sleep_prepared
            .dropna(subset=["uid_clean", "date"])
            .groupby(["uid_clean", "date"], as_index=False)[sleep_value_columns]
            .mean()
        )

    if uid_col is not None:
        sleep_user = (
            sleep_prepared
            .dropna(subset=["uid_clean"])
            .groupby("uid_clean", as_index=False)[sleep_value_columns]
            .mean()
        )

    print("\nSleep daily shape:", None if sleep_daily is None else sleep_daily.shape)
    print("Sleep user shape:", None if sleep_user is None else sleep_user.shape)


# ------------------------------------------------------------
# phonelock.Rds
# ------------------------------------------------------------

screen_daily = None
screen_user = None

if "phonelock.Rds" in loaded:
    phonelock_raw = loaded["phonelock.Rds"].copy()

    uid_col = guess_uid_column(phonelock_raw)
    time_col = guess_time_column(phonelock_raw)

    print("\nPhonelock columns detected:")
    print("uid:", uid_col)
    print("time:", time_col)

    phonelock_prepared = phonelock_raw.copy()

    if uid_col is not None:
        phonelock_prepared["uid_clean"] = phonelock_prepared[uid_col].astype(str)

    if time_col is not None:
        phonelock_prepared = add_date_column(phonelock_prepared, time_col)

    # Try to find a duration column
    duration_col = find_column(
        phonelock_prepared,
        contains_any=["duration", "seconds", "minutes", "hours"]
    )

    if duration_col is not None:
        phonelock_prepared["phone_use_proxy"] = safe_numeric(phonelock_prepared[duration_col])
        proxy_description = f"numeric duration column: {duration_col}"
    else:
        # Fallback: event count per day/user
        phonelock_prepared["phone_use_proxy"] = 1
        proxy_description = "phone lock event count"

    print("phone_use_proxy based on:", proxy_description)

    if uid_col is not None and "date" in phonelock_prepared.columns:
        screen_daily = (
            phonelock_prepared
            .dropna(subset=["uid_clean", "date"])
            .groupby(["uid_clean", "date"], as_index=False)["phone_use_proxy"]
            .sum()
        )

    if uid_col is not None:
        screen_user = (
            phonelock_prepared
            .dropna(subset=["uid_clean"])
            .groupby("uid_clean", as_index=False)["phone_use_proxy"]
            .sum()
        )

    print("Screen daily shape:", None if screen_daily is None else screen_daily.shape)
    print("Screen user shape:", None if screen_user is None else screen_user.shape)


# ------------------------------------------------------------
# Mood.Rds
# ------------------------------------------------------------

mood_daily = None
mood_user = None

if "Mood.Rds" in loaded:
    mood_raw = loaded["Mood.Rds"].copy()

    uid_col = guess_uid_column(mood_raw)
    time_col = guess_time_column(mood_raw)

    happy_col = find_column(mood_raw, possible_names=["happy"], contains_any=["happy"])
    sad_col = find_column(mood_raw, possible_names=["sad"], contains_any=["sad"])

    print("\nMood columns detected:")
    print("uid:", uid_col)
    print("time:", time_col)
    print("happy:", happy_col)
    print("sad:", sad_col)

    mood_prepared = mood_raw.copy()

    if uid_col is not None:
        mood_prepared["uid_clean"] = mood_prepared[uid_col].astype(str)

    if time_col is not None:
        mood_prepared = add_date_column(mood_prepared, time_col)

    if happy_col is not None and sad_col is not None:
        mood_prepared["happy_numeric"] = safe_numeric(mood_prepared[happy_col])
        mood_prepared["sad_numeric"] = safe_numeric(mood_prepared[sad_col])

        # Same earlier project mood score logic
        mood_prepared["mood_score"] = (
            (mood_prepared["happy_numeric"] - mood_prepared["sad_numeric"] + 3) / 6
        ) * 100
    else:
        # Fallback: first numeric mood-like column
        numeric_columns = mood_prepared.select_dtypes(include="number").columns.tolist()

        if len(numeric_columns) > 0:
            mood_prepared["mood_score"] = mood_prepared[numeric_columns[0]]

    if "mood_score" in mood_prepared.columns and uid_col is not None and "date" in mood_prepared.columns:
        mood_daily = (
            mood_prepared
            .dropna(subset=["uid_clean", "date"])
            .groupby(["uid_clean", "date"], as_index=False)["mood_score"]
            .mean()
        )

    if "mood_score" in mood_prepared.columns and uid_col is not None:
        mood_user = (
            mood_prepared
            .dropna(subset=["uid_clean"])
            .groupby("uid_clean", as_index=False)["mood_score"]
            .mean()
        )

    print("Mood daily shape:", None if mood_daily is None else mood_daily.shape)
    print("Mood user shape:", None if mood_user is None else mood_user.shape)


# ------------------------------------------------------------
# PerceivedStressScale.Rds
# ------------------------------------------------------------

stress_user = None

if "PerceivedStressScale.Rds" in loaded:
    pss_raw = loaded["PerceivedStressScale.Rds"].copy()

    uid_col = guess_uid_column(pss_raw)

    print("\nPSS columns detected:")
    print("uid:", uid_col)
    print(list(pss_raw.columns))

    pss_prepared = pss_raw.copy()

    if uid_col is not None:
        pss_prepared["uid_clean"] = pss_prepared[uid_col].astype(str)

    response_map = {
        "never": 0,
        "almost never": 1,
        "sometime": 2,
        "sometimes": 2,
        "fairly often": 3,
        "very often": 4
    }

    question_columns = [
        column for column in pss_prepared.columns
        if normalize_column_name(column).startswith("q")
    ]

    scored_columns = []

    for column in question_columns:
        clean_values = (
            pss_prepared[column]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        numeric = clean_values.map(response_map)

        # Also allow already numeric values
        numeric = numeric.fillna(safe_numeric(pss_prepared[column]))

        scored_column = f"{column}_score"
        pss_prepared[scored_column] = numeric
        scored_columns.append(scored_column)

    # Reverse-score PSS items 4, 5, 7, 8 if they exist
    reverse_question_numbers = ["q4", "q5", "q7", "q8"]

    for scored_column in scored_columns:
        clean_scored = normalize_column_name(scored_column)

        if any(clean_scored.startswith(reverse_q) for reverse_q in reverse_question_numbers):
            pss_prepared[scored_column] = 4 - pss_prepared[scored_column]

    if len(scored_columns) > 0:
        pss_prepared["stress_score"] = pss_prepared[scored_columns].sum(axis=1)

        if uid_col is not None:
            stress_user = (
                pss_prepared
                .dropna(subset=["uid_clean"])
                .groupby("uid_clean", as_index=False)["stress_score"]
                .mean()
            )

    print("Stress user shape:", None if stress_user is None else stress_user.shape)

Sleep columns detected:
uid: uid
time: timestamp
rate: rate
hour: hour

Sleep daily shape: (49, 5)
Sleep user shape: (49, 4)

Phonelock columns detected:
uid: uid
time: start_timestamp
phone_use_proxy based on: phone lock event count
Screen daily shape: (49, 3)
Screen user shape: (49, 2)

Mood columns detected:
uid: uid
time: timestamp
happy: happy
sad: sad
Mood daily shape: (38, 3)
Mood user shape: (38, 2)

PSS columns detected:
uid: uid
['uid', 'type', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10']
Stress user shape: (46, 2)


## 5. Run StudentLife Bridge Tests

This step tests:

A. screen use → sleep quality
B. sleep quality → mood
C. sleep quality → stress

Each test compares:

baseline mean model
linear regression
random forest comparison

A path is only marked usable if it beats the baseline mean model on test RMSE.

In [6]:
# ------------------------------------------------------------
# Run StudentLife bridge tests
# ------------------------------------------------------------

studentlife_results = []

# ------------------------------------------------------------
# A. Screen use → sleep quality
# ------------------------------------------------------------

if screen_daily is not None and sleep_daily is not None:
    screen_sleep_data = pd.merge(
        screen_daily,
        sleep_daily,
        on=["uid_clean", "date"],
        how="inner"
    )

    print("Screen → sleep daily merge shape:", screen_sleep_data.shape)

    if "sleep_quality_problem" in screen_sleep_data.columns:
        result = evaluate_regression_path(
            path_name="A: screen use → sleep quality",
            data=screen_sleep_data,
            predictors=["phone_use_proxy"],
            target="sleep_quality_problem",
            output_label="sleep quality",
            graph_prefix="studentlife_screen_to_sleep_quality"
        )

        studentlife_results.append(result)

elif screen_user is not None and sleep_user is not None:
    screen_sleep_data = pd.merge(
        screen_user,
        sleep_user,
        on="uid_clean",
        how="inner"
    )

    print("Screen → sleep user merge shape:", screen_sleep_data.shape)

    if "sleep_quality_problem" in screen_sleep_data.columns:
        result = evaluate_regression_path(
            path_name="A: screen use → sleep quality",
            data=screen_sleep_data,
            predictors=["phone_use_proxy"],
            target="sleep_quality_problem",
            output_label="sleep quality",
            graph_prefix="studentlife_screen_to_sleep_quality"
        )

        studentlife_results.append(result)

else:
    print("Could not test screen use → sleep quality because screen or sleep data was missing.")


# ------------------------------------------------------------
# B. Sleep quality → mood
# ------------------------------------------------------------

if sleep_daily is not None and mood_daily is not None:
    sleep_mood_data = pd.merge(
        sleep_daily,
        mood_daily,
        on=["uid_clean", "date"],
        how="inner"
    )

    print("Sleep → mood daily merge shape:", sleep_mood_data.shape)

    predictors = [
        column for column in ["sleep_quality_problem", "sleep_hours"]
        if column in sleep_mood_data.columns
    ]

    if "mood_score" in sleep_mood_data.columns and len(predictors) > 0:
        result = evaluate_regression_path(
            path_name="B: sleep quality → mood",
            data=sleep_mood_data,
            predictors=predictors,
            target="mood_score",
            output_label="mood",
            graph_prefix="studentlife_sleep_to_mood"
        )

        studentlife_results.append(result)

elif sleep_user is not None and mood_user is not None:
    sleep_mood_data = pd.merge(
        sleep_user,
        mood_user,
        on="uid_clean",
        how="inner"
    )

    print("Sleep → mood user merge shape:", sleep_mood_data.shape)

    predictors = [
        column for column in ["sleep_quality_problem", "sleep_hours"]
        if column in sleep_mood_data.columns
    ]

    if "mood_score" in sleep_mood_data.columns and len(predictors) > 0:
        result = evaluate_regression_path(
            path_name="B: sleep quality → mood",
            data=sleep_mood_data,
            predictors=predictors,
            target="mood_score",
            output_label="mood",
            graph_prefix="studentlife_sleep_to_mood"
        )

        studentlife_results.append(result)

else:
    print("Could not test sleep quality → mood because sleep or mood data was missing.")


# ------------------------------------------------------------
# C. Sleep quality → stress
# ------------------------------------------------------------

if sleep_user is not None and stress_user is not None:
    sleep_stress_data = pd.merge(
        sleep_user,
        stress_user,
        on="uid_clean",
        how="inner"
    )

    print("Sleep → stress user merge shape:", sleep_stress_data.shape)

    predictors = [
        column for column in ["sleep_quality_problem", "sleep_hours"]
        if column in sleep_stress_data.columns
    ]

    if "stress_score" in sleep_stress_data.columns and len(predictors) > 0:
        result = evaluate_regression_path(
            path_name="C: sleep quality → stress",
            data=sleep_stress_data,
            predictors=predictors,
            target="stress_score",
            output_label="stress",
            graph_prefix="studentlife_sleep_to_stress"
        )

        studentlife_results.append(result)

else:
    print("Could not test sleep quality → stress because sleep or PSS data was missing.")


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

studentlife_results_table = pd.DataFrame(studentlife_results)

display(studentlife_results_table)

# Save results
studentlife_results_path = PROJECT_DIR / "studentlife_bridge_model_results.csv"
studentlife_results_table.to_csv(studentlife_results_path, index=False)

print("Saved StudentLife results to:")
print(studentlife_results_path)

Screen → sleep daily merge shape: (49, 6)
Sleep → mood daily merge shape: (38, 6)
Sleep → stress user merge shape: (46, 5)


,path,output,target,rows,baseline_r2,baseline_mae,baseline_rmse,linear_train_r2,linear_test_r2,linear_mae,linear_rmse,comparison_model,comparison_train_r2,comparison_test_r2,comparison_mae,comparison_rmse,best_model,beats_baseline,decision,graph_path
0,A: screen use → sleep quality,sleep quality,sleep_quality_problem,49,-0.2858,0.4098,0.5148,0.0039,-0.2685,0.4053,0.5113,random forest,0.2876,-0.2543,0.3856,0.5085,random forest,Yes,use with caution,D:\Documentos\Sleep datasets\sleep_installatio...
1,B: sleep quality → mood,mood,mood_score,38,-0.0024,18.4183,21.2411,0.0489,0.0169,17.8291,21.0353,random forest,0.3408,-0.1166,19.9220,22.4185,linear regression,Yes,use with caution,D:\Documentos\Sleep datasets\sleep_installatio...
2,C: sleep quality → stress,stress,stress_score,46,-0.0013,4.8750,6.2771,0.0935,0.3306,4.2257,5.1326,random forest,0.3470,0.1355,4.9421,5.8327,linear regression,Yes,use with caution,D:\Documentos\Sleep datasets\sleep_installatio...


Saved StudentLife results to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\studentlife_bridge_model_results.csv


## 6. Final StudentLife Bridge Decision

This step writes a concise summary into model_results_summary.md.

The key decision is whether StudentLife supports:

screen time → sleep quality
sleep quality → mood
sleep quality → stress

In [7]:
# ------------------------------------------------------------
# Write summary to model_results_summary.md
# ------------------------------------------------------------

summary_path = PROJECT_DIR / "model_results_summary.md"

if len(studentlife_results_table) == 0:
    studentlife_summary_text = """
## StudentLife Bridge Tests — Actual Local Results

The StudentLife files were found locally, but no valid bridge tests could be completed.

Decision:

- Screen time → sleep quality: **not tested**
- Sleep quality → mood: **not tested**
- Sleep quality → stress: **not tested**

Reason:

The required variables or join keys were not available in a usable merged table.

No connection is claimed without a successful dataset test.
"""
else:
    usable_rows = studentlife_results_table[
        studentlife_results_table["decision"] == "use with caution"
    ]

    studentlife_summary_text = "\n## StudentLife Bridge Tests — Actual Local Results\n\n"

    studentlife_summary_text += "StudentLife was tested locally using available RDS files.\n\n"

    studentlife_summary_text += "Decision rule:\n\n"
    studentlife_summary_text += "- **Use with caution** only if the best tested model has lower test RMSE than the baseline mean model.\n"
    studentlife_summary_text += "- **Do not use** if the model does not beat baseline.\n"
    studentlife_summary_text += "- `sleep_score` was **not** used as evidence.\n\n"

    studentlife_summary_text += "### Results table\n\n"
    studentlife_summary_text += studentlife_results_table.to_markdown(index=False)
    studentlife_summary_text += "\n\n"

    studentlife_summary_text += "### Bridge decision\n\n"

    if len(usable_rows) == 0:
        studentlife_summary_text += "No StudentLife bridge path beat baseline. StudentLife does not currently support the sleep-quality bridge with the tested variables.\n"
    else:
        studentlife_summary_text += "The following StudentLife paths beat baseline and may be used with caution:\n\n"

        for _, row in usable_rows.iterrows():
            studentlife_summary_text += f"- {row['path']} → {row['output']} using best model: {row['best_model']}\n"

    studentlife_summary_text += "\n### Limitations\n\n"
    studentlife_summary_text += "- StudentLife has a small participant sample.\n"
    studentlife_summary_text += "- Daily joins may lose rows when entries do not occur on the same date.\n"
    studentlife_summary_text += "- Phone lock data is a proxy for screen/phone use, not perfect measured screen time.\n"
    studentlife_summary_text += "- These models are observational and do not prove causality.\n"

with open(summary_path, "a", encoding="utf-8") as file:
    file.write("\n\n")
    file.write(studentlife_summary_text)

print("Updated summary:")
print(summary_path)

print("\nSummary preview:")
print(studentlife_summary_text[:3000])

Updated summary:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\model_results_summary.md

Summary preview:

## StudentLife Bridge Tests — Actual Local Results

StudentLife was tested locally using available RDS files.

Decision rule:

- **Use with caution** only if the best tested model has lower test RMSE than the baseline mean model.
- **Do not use** if the model does not beat baseline.
- `sleep_score` was **not** used as evidence.

### Results table

| path                          | output        | target                |   rows |   baseline_r2 |   baseline_mae |   baseline_rmse |   linear_train_r2 |   linear_test_r2 |   linear_mae |   linear_rmse | comparison_model   |   comparison_train_r2 |   comparison_test_r2 |   comparison_mae |   comparison_rmse | best_model        | beats_baseline   | decision         | graph_path                                                                                                                       |
|:-------------------------

## 7. Extract StudentLife Linear Regression Formulas

The previous step tested whether each StudentLife bridge path beat the baseline model.

This step extracts interpretable linear regression formulas for the usable StudentLife paths.

The formulas are provided in two forms:

standardized formula
raw-value formula

The standardized formula uses z-scores:

z(value) = (value - training mean) / training standard deviation

The raw-value formula can be used more directly in the installation code.

Important:

sleep_score is not used as evidence.

These are observational models, so they do not prove causality.

In [8]:
# ------------------------------------------------------------
# Extract interpretable StudentLife linear regression formulas
# ------------------------------------------------------------

from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd


def fit_linear_formula_model(path_name, data, predictors, target):
    """
    Fit a standardized linear regression model and convert it into:
    1. standardized coefficient formula
    2. raw-value coefficient formula

    This uses the same strict baseline comparison logic:
    usable only if the linear model has lower test RMSE than the baseline mean model.
    """

    model_data = data[predictors + [target]].copy()

    for column in predictors + [target]:
        model_data[column] = pd.to_numeric(model_data[column], errors="coerce")

    model_data = model_data.dropna().copy()

    if len(model_data) < 20:
        return None, None

    X = model_data[predictors]
    y = model_data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42
    )

    # --------------------------------------------------------
    # Baseline model
    # --------------------------------------------------------

    baseline_model = DummyRegressor(strategy="mean")
    baseline_model.fit(X_train, y_train)

    baseline_test_predictions = baseline_model.predict(X_test)

    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_test_predictions))
    baseline_mae = mean_absolute_error(y_test, baseline_test_predictions)
    baseline_r2 = r2_score(y_test, baseline_test_predictions)

    # --------------------------------------------------------
    # Standardized linear regression
    # --------------------------------------------------------

    linear_pipeline = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    linear_pipeline.fit(X_train, y_train)

    linear_train_predictions = linear_pipeline.predict(X_train)
    linear_test_predictions = linear_pipeline.predict(X_test)

    linear_train_r2 = r2_score(y_train, linear_train_predictions)
    linear_test_r2 = r2_score(y_test, linear_test_predictions)
    linear_mae = mean_absolute_error(y_test, linear_test_predictions)
    linear_rmse = np.sqrt(mean_squared_error(y_test, linear_test_predictions))

    beats_baseline = linear_rmse < baseline_rmse

    scaler = linear_pipeline.named_steps["scaler"]
    linear_model = linear_pipeline.named_steps["model"]

    standardized_intercept = linear_model.intercept_
    standardized_coefficients = linear_model.coef_

    feature_means = scaler.mean_
    feature_scales = scaler.scale_

    # --------------------------------------------------------
    # Convert standardized formula into raw-value formula
    # --------------------------------------------------------
    # standardized model:
    # y = intercept + sum(beta_i * z(x_i))
    #
    # z(x_i) = (x_i - mean_i) / sd_i
    #
    # raw model:
    # y = raw_intercept + sum(raw_beta_i * x_i)
    # --------------------------------------------------------

    raw_coefficients = standardized_coefficients / feature_scales

    raw_intercept = standardized_intercept - np.sum(
        standardized_coefficients * feature_means / feature_scales
    )

    # --------------------------------------------------------
    # Build formula strings
    # --------------------------------------------------------

    standardized_parts = [
        f"{standardized_coefficients[i]:+.4f} * z({predictors[i]})"
        for i in range(len(predictors))
    ]

    raw_parts = [
        f"{raw_coefficients[i]:+.4f} * {predictors[i]}"
        for i in range(len(predictors))
    ]

    standardized_formula = (
        f"{target} = {standardized_intercept:.4f} "
        + " ".join(standardized_parts)
    )

    raw_formula = (
        f"{target} = {raw_intercept:.4f} "
        + " ".join(raw_parts)
    )

    # --------------------------------------------------------
    # Formula summary row
    # --------------------------------------------------------

    formula_summary = {
        "path": path_name,
        "target": target,
        "predictors": ", ".join(predictors),
        "rows_used": len(model_data),
        "baseline_test_r2": round(baseline_r2, 4),
        "baseline_test_mae": round(baseline_mae, 4),
        "baseline_test_rmse": round(baseline_rmse, 4),
        "linear_train_r2": round(linear_train_r2, 4),
        "linear_test_r2": round(linear_test_r2, 4),
        "linear_test_mae": round(linear_mae, 4),
        "linear_test_rmse": round(linear_rmse, 4),
        "beats_baseline": "Yes" if beats_baseline else "No",
        "decision": "use with caution" if beats_baseline else "do not use",
        "standardized_formula": standardized_formula,
        "raw_formula": raw_formula
    }

    # --------------------------------------------------------
    # Coefficient table
    # --------------------------------------------------------

    coefficient_rows = []

    coefficient_rows.append({
        "path": path_name,
        "target": target,
        "term": "Intercept",
        "standardized_coefficient": standardized_intercept,
        "raw_coefficient": raw_intercept,
        "training_mean": "",
        "training_sd": ""
    })

    for i, predictor in enumerate(predictors):
        coefficient_rows.append({
            "path": path_name,
            "target": target,
            "term": predictor,
            "standardized_coefficient": standardized_coefficients[i],
            "raw_coefficient": raw_coefficients[i],
            "training_mean": feature_means[i],
            "training_sd": feature_scales[i]
        })

    coefficient_table = pd.DataFrame(coefficient_rows)

    return formula_summary, coefficient_table


# ------------------------------------------------------------
# Define StudentLife formula paths
# ------------------------------------------------------------

formula_jobs = []

# A. screen use → sleep quality
if "screen_sleep_data" in globals():
    if "phone_use_proxy" in screen_sleep_data.columns and "sleep_quality_problem" in screen_sleep_data.columns:
        formula_jobs.append({
            "path_name": "A: screen use → sleep quality",
            "data": screen_sleep_data,
            "predictors": ["phone_use_proxy"],
            "target": "sleep_quality_problem"
        })

# B. sleep quality → mood
if "sleep_mood_data" in globals():
    mood_predictors = [
        column for column in ["sleep_quality_problem", "sleep_hours"]
        if column in sleep_mood_data.columns
    ]

    if "mood_score" in sleep_mood_data.columns and len(mood_predictors) > 0:
        formula_jobs.append({
            "path_name": "B: sleep quality → mood",
            "data": sleep_mood_data,
            "predictors": mood_predictors,
            "target": "mood_score"
        })

# C. sleep quality → stress
if "sleep_stress_data" in globals():
    stress_predictors = [
        column for column in ["sleep_quality_problem", "sleep_hours"]
        if column in sleep_stress_data.columns
    ]

    if "stress_score" in sleep_stress_data.columns and len(stress_predictors) > 0:
        formula_jobs.append({
            "path_name": "C: sleep quality → stress",
            "data": sleep_stress_data,
            "predictors": stress_predictors,
            "target": "stress_score"
        })


# ------------------------------------------------------------
# Fit formulas
# ------------------------------------------------------------

formula_summaries = []
coefficient_tables = []

for job in formula_jobs:
    formula_summary, coefficient_table = fit_linear_formula_model(
        path_name=job["path_name"],
        data=job["data"],
        predictors=job["predictors"],
        target=job["target"]
    )

    if formula_summary is not None:
        formula_summaries.append(formula_summary)
        coefficient_tables.append(coefficient_table)

studentlife_formula_summary_table = pd.DataFrame(formula_summaries)

if len(coefficient_tables) > 0:
    studentlife_coefficient_table = pd.concat(coefficient_tables, ignore_index=True)
else:
    studentlife_coefficient_table = pd.DataFrame()

print("StudentLife formula summary:")
display(studentlife_formula_summary_table)

print("StudentLife coefficient table:")
display(studentlife_coefficient_table)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

formula_summary_path = PROJECT_DIR / "studentlife_installation_formula_summary.csv"
coefficient_path = PROJECT_DIR / "studentlife_installation_formula_coefficients.csv"

studentlife_formula_summary_table.to_csv(formula_summary_path, index=False)
studentlife_coefficient_table.to_csv(coefficient_path, index=False)

print("Saved formula summary to:")
print(formula_summary_path)

print("\nSaved coefficient table to:")
print(coefficient_path)

StudentLife formula summary:


,path,target,predictors,rows_used,baseline_test_r2,baseline_test_mae,baseline_test_rmse,linear_train_r2,linear_test_r2,linear_test_mae,linear_test_rmse,beats_baseline,decision,standardized_formula,raw_formula
0,A: screen use → sleep quality,sleep_quality_problem,phone_use_proxy,49,-0.2858,0.4098,0.5148,0.0039,-0.2685,0.4053,0.5113,Yes,use with caution,sleep_quality_problem = 0.9224 +0.0238 * z(pho...,sleep_quality_problem = 0.8633 +0.0003 * phone...
1,B: sleep quality → mood,mood_score,"sleep_quality_problem, sleep_hours",38,-0.0024,18.4183,21.2411,0.0489,0.0169,17.8291,21.0353,Yes,use with caution,mood_score = 55.3811 +1.2606 * z(sleep_quality...,mood_score = 27.6911 +3.7101 * sleep_quality_p...
2,C: sleep quality → stress,stress_score,"sleep_quality_problem, sleep_hours",46,-0.0013,4.8750,6.2771,0.0935,0.3306,4.2257,5.1326,Yes,use with caution,stress_score = 18.2353 +0.4816 * z(sleep_quali...,stress_score = 31.1402 +1.1541 * sleep_quality...


StudentLife coefficient table:


,path,target,term,standardized_coefficient,raw_coefficient,training_mean,training_sd
0,A: screen use → sleep quality,sleep_quality_problem,Intercept,0.922430,0.863320,,
1,A: screen use → sleep quality,sleep_quality_problem,phone_use_proxy,0.023791,0.000313,188.583333,75.903219
2,B: sleep quality → mood,mood_score,Intercept,55.381101,27.691094,,
3,B: sleep quality → mood,mood_score,sleep_quality_problem,1.260607,3.710053,0.917641,0.339781
4,B: sleep quality → mood,mood_score,sleep_hours,2.624049,3.284802,7.393294,0.798845
5,C: sleep quality → stress,stress_score,Intercept,18.235294,31.140160,,
6,C: sleep quality → stress,stress_score,sleep_quality_problem,0.481575,1.154060,0.881137,0.417288
7,C: sleep quality → stress,stress_score,sleep_hours,-1.666836,-1.945134,7.157218,0.856926


Saved formula summary to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\studentlife_installation_formula_summary.csv

Saved coefficient table to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\studentlife_installation_formula_coefficients.csv


## 8. Save StudentLife Installation Formula Summary

This step appends the final interpretable formulas to model_results_summary.md.

These formulas are meant for implementation discussion with the installation team.

They should be used with caution because the models are observational and the StudentLife sample is small.

In [9]:
# ------------------------------------------------------------
# Append StudentLife formulas to model_results_summary.md
# ------------------------------------------------------------

summary_path = PROJECT_DIR / "model_results_summary.md"

formula_text = "\n\n## StudentLife Installation Formulas\n\n"

formula_text += "These formulas were extracted from the linear regression models.\n\n"
formula_text += "They are provided for installation implementation.\n\n"
formula_text += "`sleep_score` was not used as evidence.\n\n"
formula_text += "A formula is marked usable only if the linear regression model beats the baseline mean model on test RMSE.\n\n"

if len(studentlife_formula_summary_table) == 0:
    formula_text += "No StudentLife formulas could be extracted.\n"

else:
    for _, row in studentlife_formula_summary_table.iterrows():
        formula_text += f"### {row['path']}\n\n"
        formula_text += f"- Target: `{row['target']}`\n"
        formula_text += f"- Predictors: `{row['predictors']}`\n"
        formula_text += f"- Rows used: `{row['rows_used']}`\n"
        formula_text += f"- Baseline RMSE: `{row['baseline_test_rmse']}`\n"
        formula_text += f"- Linear RMSE: `{row['linear_test_rmse']}`\n"
        formula_text += f"- Linear test R²: `{row['linear_test_r2']}`\n"
        formula_text += f"- Decision: **{row['decision']}**\n\n"

        formula_text += "Standardized formula:\n\n"
        formula_text += "```text\n"
        formula_text += row["standardized_formula"]
        formula_text += "\n```\n\n"

        formula_text += "Raw-value formula:\n\n"
        formula_text += "```text\n"
        formula_text += row["raw_formula"]
        formula_text += "\n```\n\n"

formula_text += "### Important interpretation notes\n\n"
formula_text += "- `sleep_quality_problem` means worse sleep quality when the value is higher.\n"
formula_text += "- `mood_score` means better mood when the value is higher.\n"
formula_text += "- `stress_score` means more stress when the value is higher.\n"
formula_text += "- These formulas are observational and do not prove causality.\n"
formula_text += "- Use them with caution because StudentLife has a small sample and the usable paths only modestly beat baseline.\n"

with open(summary_path, "a", encoding="utf-8") as file:
    file.write(formula_text)

print("Updated summary:")
print(summary_path)

print("\nFormula summary preview:")
print(formula_text[:4000])

Updated summary:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\model_results_summary.md

Formula summary preview:


## StudentLife Installation Formulas

These formulas were extracted from the linear regression models.

They are provided for installation implementation.

`sleep_score` was not used as evidence.

A formula is marked usable only if the linear regression model beats the baseline mean model on test RMSE.

### A: screen use → sleep quality

- Target: `sleep_quality_problem`
- Predictors: `phone_use_proxy`
- Rows used: `49`
- Baseline RMSE: `0.5148`
- Linear RMSE: `0.5113`
- Linear test R²: `-0.2685`
- Decision: **use with caution**

Standardized formula:

```text
sleep_quality_problem = 0.9224 +0.0238 * z(phone_use_proxy)
```

Raw-value formula:

```text
sleep_quality_problem = 0.8633 +0.0003 * phone_use_proxy
```

### B: sleep quality → mood

- Target: `mood_score`
- Predictors: `sleep_quality_problem, sleep_hours`
- Rows used: `38`
- Baseline RMSE: `21.2411

## 9. Test Output Models Without Sleep Hours

The previous StudentLife output models used both sleep quality and sleep hours.

However, the installation does not collect sleep hours.

This step tests a cleaner bridge:

sleep_quality_problem → mood
sleep_quality_problem → stress

This tells us whether sleep quality alone can connect to the fixed outputs.

In [10]:
# ------------------------------------------------------------
# Test StudentLife output models using only sleep quality
# ------------------------------------------------------------

# Safety check: make sure earlier notebook cells were run
required_objects = [
    "evaluate_regression_path",
    "sleep_mood_data",
    "sleep_stress_data",
    "PROJECT_DIR"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if len(missing_objects) > 0:
    print("Missing required objects:")
    for name in missing_objects:
        print(f"- {name}")

    print("\nPlease run the earlier StudentLife notebook cells first, from the setup cell down to the model test cell.")
    print("Then run this cell again.")

else:
    sleep_quality_only_results = []

    # --------------------------------------------------------
    # B2. Sleep quality only → mood
    # --------------------------------------------------------

    if "sleep_quality_problem" in sleep_mood_data.columns and "mood_score" in sleep_mood_data.columns:
        mood_result = evaluate_regression_path(
            path_name="B2: sleep quality only → mood",
            data=sleep_mood_data,
            predictors=["sleep_quality_problem"],
            target="mood_score",
            output_label="mood",
            graph_prefix="studentlife_sleep_quality_only_to_mood"
        )

        sleep_quality_only_results.append(mood_result)

    else:
        print("Could not test mood because needed columns were missing.")
        print("Needed: sleep_quality_problem and mood_score")

    # --------------------------------------------------------
    # C2. Sleep quality only → stress
    # --------------------------------------------------------

    if "sleep_quality_problem" in sleep_stress_data.columns and "stress_score" in sleep_stress_data.columns:
        stress_result = evaluate_regression_path(
            path_name="C2: sleep quality only → stress",
            data=sleep_stress_data,
            predictors=["sleep_quality_problem"],
            target="stress_score",
            output_label="stress",
            graph_prefix="studentlife_sleep_quality_only_to_stress"
        )

        sleep_quality_only_results.append(stress_result)

    else:
        print("Could not test stress because needed columns were missing.")
        print("Needed: sleep_quality_problem and stress_score")

    # --------------------------------------------------------
    # Show and save results
    # --------------------------------------------------------

    sleep_quality_only_results_table = pd.DataFrame(sleep_quality_only_results)

    display(sleep_quality_only_results_table)

    output_path = PROJECT_DIR / "studentlife_sleep_quality_only_output_results.csv"
    sleep_quality_only_results_table.to_csv(output_path, index=False)

    print("Saved sleep-quality-only results to:")
    print(output_path)

,path,output,target,rows,baseline_r2,baseline_mae,baseline_rmse,linear_train_r2,linear_test_r2,linear_mae,linear_rmse,comparison_model,comparison_train_r2,comparison_test_r2,comparison_mae,comparison_rmse,best_model,beats_baseline,decision,graph_path
0,B2: sleep quality only → mood,mood,mood_score,38,-0.0024,18.4183,21.2411,0.0015,-0.0289,18.5378,21.5204,random forest,0.2698,-0.0528,19.4862,21.7692,baseline mean,No,do not use,D:\Documentos\Sleep datasets\sleep_installatio...
1,C2: sleep quality only → stress,stress,stress_score,46,-0.0013,4.8750,6.2771,0.0418,0.0326,5.0238,6.1702,random forest,0.2084,-0.0163,5.5402,6.3240,linear regression,Yes,use with caution,D:\Documentos\Sleep datasets\sleep_installatio...


Saved sleep-quality-only results to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\studentlife_sleep_quality_only_output_results.csv


In [11]:
# ------------------------------------------------------------
# Show the important sleep-quality-only decision columns
# ------------------------------------------------------------

important_columns = [
    "path",
    "output",
    "target",
    "rows",
    "baseline_rmse",
    "linear_rmse",
    "comparison_model",
    "comparison_rmse",
    "best_model",
    "beats_baseline",
    "decision"
]

available_columns = [
    column for column in important_columns
    if column in sleep_quality_only_results_table.columns
]

sleep_quality_only_decision_table = sleep_quality_only_results_table[available_columns].copy()

display(sleep_quality_only_decision_table)

print("Sleep-quality-only decisions:")

for _, row in sleep_quality_only_decision_table.iterrows():
    print("\n" + row["path"])
    print(f"Baseline RMSE: {row['baseline_rmse']}")

    if "linear_rmse" in row:
        print(f"Linear RMSE: {row['linear_rmse']}")

    if "comparison_rmse" in row:
        print(f"Comparison RMSE: {row['comparison_rmse']}")

    print(f"Best model: {row['best_model']}")
    print(f"Beats baseline: {row['beats_baseline']}")
    print(f"Decision: {row['decision']}")

,path,output,target,rows,baseline_rmse,linear_rmse,comparison_model,comparison_rmse,best_model,beats_baseline,decision
0,B2: sleep quality only → mood,mood,mood_score,38,21.2411,21.5204,random forest,21.7692,baseline mean,No,do not use
1,C2: sleep quality only → stress,stress,stress_score,46,6.2771,6.1702,random forest,6.3240,linear regression,Yes,use with caution


Sleep-quality-only decisions:

B2: sleep quality only → mood
Baseline RMSE: 21.2411
Linear RMSE: 21.5204
Comparison RMSE: 21.7692
Best model: baseline mean
Beats baseline: No
Decision: do not use

C2: sleep quality only → stress
Baseline RMSE: 6.2771
Linear RMSE: 6.1702
Comparison RMSE: 6.324
Best model: linear regression
Beats baseline: Yes
Decision: use with caution


## 10. Extract Clean Stress Formula Without Sleep Hours

The sleep-quality-only test showed that mood does not beat baseline without sleep hours.

Stress does beat baseline without sleep hours.

Therefore, the cleanest StudentLife output formula is:

sleep_quality_problem → stress_score

This formula can connect to the fixed installation because it does not require sleep hours.

In [12]:
# ------------------------------------------------------------
# Extract clean stress formula using only sleep quality
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

required_objects = [
    "sleep_stress_data",
    "PROJECT_DIR"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if len(missing_objects) > 0:
    print("Missing required objects:")
    for name in missing_objects:
        print(f"- {name}")

    print("\nPlease run the earlier StudentLife cells first.")

else:
    clean_stress_data = sleep_stress_data[
        ["sleep_quality_problem", "stress_score"]
    ].copy()

    clean_stress_data["sleep_quality_problem"] = pd.to_numeric(
        clean_stress_data["sleep_quality_problem"],
        errors="coerce"
    )

    clean_stress_data["stress_score"] = pd.to_numeric(
        clean_stress_data["stress_score"],
        errors="coerce"
    )

    clean_stress_data = clean_stress_data.dropna().copy()

    X = clean_stress_data[["sleep_quality_problem"]]
    y = clean_stress_data["stress_score"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42
    )

    # --------------------------------------------------------
    # Baseline model
    # --------------------------------------------------------

    baseline_model = DummyRegressor(strategy="mean")
    baseline_model.fit(X_train, y_train)

    baseline_predictions = baseline_model.predict(X_test)

    baseline_r2 = r2_score(y_test, baseline_predictions)
    baseline_mae = mean_absolute_error(y_test, baseline_predictions)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

    # --------------------------------------------------------
    # Linear regression with standardized predictor
    # --------------------------------------------------------

    stress_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    stress_model.fit(X_train, y_train)

    train_predictions = stress_model.predict(X_train)
    test_predictions = stress_model.predict(X_test)

    train_r2 = r2_score(y_train, train_predictions)
    test_r2 = r2_score(y_test, test_predictions)
    test_mae = mean_absolute_error(y_test, test_predictions)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

    scaler = stress_model.named_steps["scaler"]
    linear_model = stress_model.named_steps["model"]

    standardized_intercept = linear_model.intercept_
    standardized_coefficient = linear_model.coef_[0]

    predictor_mean = scaler.mean_[0]
    predictor_sd = scaler.scale_[0]

    raw_coefficient = standardized_coefficient / predictor_sd
    raw_intercept = standardized_intercept - (
        standardized_coefficient * predictor_mean / predictor_sd
    )

    beats_baseline = test_rmse < baseline_rmse

    # --------------------------------------------------------
    # Make result table
    # --------------------------------------------------------

    clean_stress_formula_table = pd.DataFrame([
        {
            "path": "C2: sleep quality only → stress",
            "target": "stress_score",
            "predictor": "sleep_quality_problem",
            "rows_used": len(clean_stress_data),
            "baseline_r2": round(baseline_r2, 4),
            "baseline_mae": round(baseline_mae, 4),
            "baseline_rmse": round(baseline_rmse, 4),
            "linear_train_r2": round(train_r2, 4),
            "linear_test_r2": round(test_r2, 4),
            "linear_mae": round(test_mae, 4),
            "linear_rmse": round(test_rmse, 4),
            "beats_baseline": "Yes" if beats_baseline else "No",
            "decision": "use with caution" if beats_baseline else "do not use",
            "standardized_formula": (
                f"stress_score = {standardized_intercept:.4f} "
                f"{standardized_coefficient:+.4f} * z(sleep_quality_problem)"
            ),
            "raw_formula": (
                f"stress_score = {raw_intercept:.4f} "
                f"{raw_coefficient:+.4f} * sleep_quality_problem"
            ),
            "training_mean_sleep_quality_problem": round(predictor_mean, 4),
            "training_sd_sleep_quality_problem": round(predictor_sd, 4)
        }
    ])

    display(clean_stress_formula_table)

    # --------------------------------------------------------
    # Save result
    # --------------------------------------------------------

    output_path = PROJECT_DIR / "studentlife_clean_stress_formula.csv"
    clean_stress_formula_table.to_csv(output_path, index=False)

    print("Saved clean stress formula to:")
    print(output_path)

    print("\nClean stress formula:")
    print(clean_stress_formula_table.loc[0, "raw_formula"])

    print("\nInterpretation:")
    print("Higher sleep_quality_problem means worse sleep quality.")
    print("Higher stress_score means more stress.")

,path,target,predictor,rows_used,baseline_r2,baseline_mae,baseline_rmse,linear_train_r2,linear_test_r2,linear_mae,linear_rmse,beats_baseline,decision,standardized_formula,raw_formula,training_mean_sleep_quality_problem,training_sd_sleep_quality_problem
0,C2: sleep quality only → stress,stress_score,sleep_quality_problem,46,-0.0013,4.875,6.2771,0.0418,0.0326,5.0238,6.1702,Yes,use with caution,stress_score = 18.2353 +1.3046 * z(sleep_quali...,stress_score = 15.4805 +3.1264 * sleep_quality...,0.8811,0.4173


Saved clean stress formula to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\studentlife_clean_stress_formula.csv

Clean stress formula:
stress_score = 15.4805 +3.1264 * sleep_quality_problem

Interpretation:
Higher sleep_quality_problem means worse sleep quality.
Higher stress_score means more stress.
